<a href="https://colab.research.google.com/github/liangwan2023/MachineLearning/blob/main/Assignment4_MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.feature_selection import SelectKBest, f_classif

import zipfile

zip_path = "Data.zip"
extract_path = "."

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [12]:

# Step 1: Feature Extraction for Breast Cancer Image Classification

#1. Reads all images from the Benign and Malignant folders
#2. Extracts texture, edge, shape, and color features from each image
#3. Stores the extracted features in a table
#4. Adds the class label: B = Benign; M = Malignant
#5. Saves the final dataset as a CSV file



# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------

# Change this path if needed
DATASET_PATH = "Data"

# Output CSV file name
OUTPUT_CSV = "breast_cancer_features.csv"

# Standard image size for all images
IMAGE_SIZE = (256, 256)


# -------------------------------------------------------------------
# FEATURE EXTRACTION FUNCTION
# -------------------------------------------------------------------

def extract_features(image_path):
    """
    Extract relevant classification features from one image.

    Features extracted:
    - Texture features:
        * grayscale mean
        * grayscale standard deviation
        * HOG mean
        * HOG standard deviation
    - Edge features:
        * edge density (Canny)
        * average edge magnitude (Sobel)
    - Shape features:
        * largest contour area
        * perimeter
        * circularity
        * aspect ratio
    - Color features:
        * mean red, green, blue values

    Parameters:
        image_path (str): Path to the image file

    Returns:
        dict: Extracted features
    """

    # Read image
    image = cv2.imread(image_path)

    # Check if image is loaded correctly
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Convert from BGR (OpenCV default) to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize all images to the same size
    image = cv2.resize(image, IMAGE_SIZE)

    # Convert image to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # ---------------------------------------------------------------
    # 1. TEXTURE FEATURES
    # ---------------------------------------------------------------

    # Histogram of Oriented Gradients (HOG)
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    gray_mean = np.mean(gray)
    gray_std = np.std(gray)
    hog_mean = np.mean(hog_features)
    hog_std = np.std(hog_features)

    # ---------------------------------------------------------------
    # 2. EDGE FEATURES
    # ---------------------------------------------------------------

    # Canny edge detection
    edges = cv2.Canny(gray, 100, 200)

    # Proportion of edge pixels
    edge_density = np.sum(edges > 0) / edges.size

    # Sobel gradient magnitude
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_magnitude = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
    edge_mean = np.mean(edge_magnitude)

    # ---------------------------------------------------------------
    # 3. SHAPE FEATURES
    # ---------------------------------------------------------------

    # Segment image using Otsu thresholding
    _, thresh = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Find external contours
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    area = 0
    perimeter = 0
    circularity = 0
    aspect_ratio = 0

    # Use the largest contour as the main object
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        perimeter = cv2.arcLength(largest_contour, True)

        # Circularity formula: 4πA / P²
        if perimeter > 0:
            circularity = (4 * np.pi * area) / (perimeter ** 2)

        # Bounding box aspect ratio
        x, y, w, h = cv2.boundingRect(largest_contour)
        if h > 0:
            aspect_ratio = w / h

    # ---------------------------------------------------------------
    # 4. COLOR FEATURES
    # ---------------------------------------------------------------

    r_mean = np.mean(image[:, :, 0])
    g_mean = np.mean(image[:, :, 1])
    b_mean = np.mean(image[:, :, 2])

    # Return all features as a dictionary
    return {
        "gray_mean": gray_mean,
        "gray_std": gray_std,
        "hog_mean": hog_mean,
        "hog_std": hog_std,
        "edge_density": edge_density,
        "edge_mean": edge_mean,
        "area": area,
        "perimeter": perimeter,
        "circularity": circularity,
        "aspect_ratio": aspect_ratio,
        "r_mean": r_mean,
        "g_mean": g_mean,
        "b_mean": b_mean
    }


# -------------------------------------------------------------------
# DATASET CREATION FUNCTION
# -------------------------------------------------------------------

def build_dataset(dataset_path):
    """
    Build a full dataset from the Benign and Malignant image folders.

    Parameters:
        dataset_path (str): Main dataset folder path

    Returns:
        pandas.DataFrame: Final feature dataset
    """

    rows = []

    # Define class folders and labels
    class_folders = {
        "Benign": "B",
        "Malignant": "M"
    }

    # Loop through each class folder
    for folder_name, class_label in class_folders.items():
        folder_path = os.path.join(dataset_path, folder_name)

        # Search for .png images
        image_paths = glob.glob(os.path.join(folder_path, "*.png"))

        # Process each image
        for image_path in image_paths:
            features = extract_features(image_path)

            # Add image name and class label
            features["image_name"] = os.path.basename(image_path)
            features["class"] = class_label

            rows.append(features)

    # Create DataFrame
    df = pd.DataFrame(rows)

    # Reorder columns so "class" appears at the end
    ordered_columns = [col for col in df.columns if col != "class"] + ["class"]
    df = df[ordered_columns]

    return df


# -------------------------------------------------------------------
# MAIN PROGRAM
# -------------------------------------------------------------------

def main():
    """
    Main function to create the dataset and save it as CSV.
    """

    print("Starting feature extraction...")
    df = build_dataset(DATASET_PATH)

    # Save dataset to CSV
    df.to_csv(OUTPUT_CSV, index=False)

    # Display summary
    print("\nFeature extraction completed successfully.")
    print(f"Dataset saved as: {OUTPUT_CSV}")
    print(f"Dataset shape: {df.shape}")
    print("\nClass distribution:")
    print(df["class"].value_counts())

    print("\nFirst 5 rows of the dataset:")
    print(df)


# Run the program
if __name__ == "__main__":
    main()

Starting feature extraction...

Feature extraction completed successfully.
Dataset saved as: breast_cancer_features.csv
Dataset shape: (61, 15)

Class distribution:
class
M    31
B    30
Name: count, dtype: int64

First 5 rows of the dataset:
     gray_mean   gray_std  hog_mean   hog_std  edge_density   edge_mean  \
0   199.341431  21.160830  0.127989  0.106755      0.079346   39.577275   
1   184.413971  32.563332  0.137435  0.094284      0.176788   79.914293   
2   177.599457  26.085522  0.148332  0.075996      0.180969   82.966010   
3   188.175797  24.086620  0.151023  0.070497      0.156326   73.870793   
4   179.500977  22.292157  0.127279  0.107600      0.098572   52.560384   
..         ...        ...       ...       ...           ...         ...   
56  215.034424  22.410123  0.152606  0.067002      0.241745  105.729585   
57  207.773758  27.416431  0.149933  0.072786      0.253723  108.951137   
58  213.768295  17.490798  0.151533  0.069394      0.196716   77.454146   
59  211

In [15]:
# Step 2 select features
"""
Feature Selection for Breast Cancer Image Classification
-------------------------------------------------------
This script:
1. Loads the extracted feature dataset from CSV
2. Separates features and class labels
3. Applies ANOVA F-test feature selection
4. Selects the top 4 most relevant features
5. Saves a new CSV file with only the selected features and class label
"""





# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

INPUT_CSV = "breast_cancer_features.csv"
OUTPUT_CSV = "breast_cancer_top4_features.csv"
TOP_K = 4

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

# Read the dataset created during feature extraction
df = pd.read_csv(INPUT_CSV)

# Remove image name column if present (not a numeric feature)
if "image_name" in df.columns:
    df = df.drop(columns=["image_name"])

# Separate input features (X) and target label (y)
X = df.drop(columns=["class"])
y = df["class"]

# Encode class labels:
# B = 0 (Benign), M = 1 (Malignant)
y_encoded = y.map({"B": 0, "M": 1})

# ------------------------------------------------------------
# FEATURE SELECTION
# ------------------------------------------------------------

# Select the top 4 features using ANOVA F-test
selector = SelectKBest(score_func=f_classif, k=TOP_K)
X_selected = selector.fit_transform(X, y_encoded)

# Get mask of selected columns
selected_mask = selector.get_support()

# Get selected feature names
selected_features = X.columns[selected_mask]

# Get scores for all features
feature_scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
})

# Sort features from highest score to lowest
feature_scores = feature_scores.sort_values(by="Score", ascending=False)

# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("Top 4 selected features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")

print("\nAll feature scores:")
print(feature_scores)

# ------------------------------------------------------------
# SAVE NEW DATASET WITH TOP 4 FEATURES
# ------------------------------------------------------------

# Create new DataFrame with selected features only
df_selected = df[selected_features.tolist()].copy()

# Add class column back
df_selected["class"] = y

# Save to CSV
df_selected.to_csv(OUTPUT_CSV, index=False)

print(f"\nNew dataset with top {TOP_K} features saved as: {OUTPUT_CSV}")
print(f"Selected dataset shape: {df_selected.shape}")

Top 4 selected features:
1. gray_mean
2. r_mean
3. g_mean
4. b_mean

All feature scores:
         Feature        Score
12        b_mean  1163.639212
10        r_mean   586.809146
0      gray_mean   402.021434
11        g_mean   142.085717
5      edge_mean    47.359176
4   edge_density    45.770069
6           area    26.748075
3        hog_std    24.236609
2       hog_mean    22.755592
8    circularity     3.578636
7      perimeter     2.960961
9   aspect_ratio     1.306107
1       gray_std     0.717166

New dataset with top 4 features saved as: breast_cancer_top4_features.csv
Selected dataset shape: (61, 5)


In [13]:
#Data Normalization for Selected Features
"""
During this step:
1. Loads the dataset with the top 4 selected features
2. Normalizes the feature values using StandardScaler
3. Saves the normalized dataset for model training
"""

import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------

INPUT_FILE = "breast_cancer_top4_features.csv"
OUTPUT_FILE = "breast_cancer_normalized.csv"

# ---------------------------------------------------
# LOAD DATA
# ---------------------------------------------------

df = pd.read_csv(INPUT_FILE)

# Separate features and class label
X = df.drop(columns=["class"])
y = df["class"]

# ---------------------------------------------------
# NORMALIZATION
# ---------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Convert back to dataframe
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Add class column again
X_scaled["class"] = y

# ---------------------------------------------------
# SAVE DATASET
# ---------------------------------------------------

X_scaled.to_csv(OUTPUT_FILE, index=False)

print("Normalization completed.")
print("Normalized dataset saved as:", OUTPUT_FILE)

print("\nFirst 5 rows of normalized data:")
print(X_scaled.head())
print("\nSummary statistics of normalized data:")
print(X_scaled.describe())

Normalization completed.
Normalized dataset saved as: breast_cancer_normalized.csv

First 5 rows of normalized data:
   gray_mean    r_mean    g_mean    b_mean class
0   0.167508 -0.475323  0.844097 -0.387784     B
1  -0.825187 -1.194896 -0.371375 -1.142702     B
2  -1.278361 -1.099541 -1.375990 -1.139771     B
3  -0.575021 -1.048914 -0.068196 -0.850048     B
4  -1.151908 -0.472608 -1.645378 -1.082717     B

Summary statistics of normalized data:
          gray_mean        r_mean        g_mean        b_mean
count  6.100000e+01  6.100000e+01  6.100000e+01  6.100000e+01
mean  -1.186665e-15 -1.110223e-16 -7.871663e-16 -7.061746e-16
std    1.008299e+00  1.008299e+00  1.008299e+00  1.008299e+00
min   -1.465511e+00 -1.491623e+00 -2.145653e+00 -1.457203e+00
25%   -1.019751e+00 -1.003194e+00 -9.971861e-01 -1.004730e+00
50%    1.675077e-01  9.664444e-02  3.825015e-01  4.591523e-01
75%    9.450212e-01  9.963964e-01  7.879868e-01  1.013651e+00
max    1.561760e+00  1.490032e+00  1.574964e+00  1.36

In [21]:
# Builds an SVM classifier

#1. Loads the normalized dataset with the selected top 4 features
#2. Splits the data into training and testing sets
#3. Builds an SVM classifier
#4. Evaluates the classifier using accuracy, confusion matrix,
 #  and classification report







# ---------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------

INPUT_FILE = "breast_cancer_normalized.csv"

# Test size: 20% for testing, 80% for training
TEST_SIZE = 0.2

# Random state for reproducibility
RANDOM_STATE = 42

# ---------------------------------------------------
# LOAD DATA
# ---------------------------------------------------

df = pd.read_csv(INPUT_FILE)

# Separate features and target
X = df.drop(columns=["class"])
y = df["class"]

# Encode class labels
# B = 0, M = 1
y = y.map({"B": 0, "M": 1})

# ---------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# ---------------------------------------------------
# BUILD SVM MODEL
# ---------------------------------------------------

# RBF kernel SVM
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=RANDOM_STATE
)

# Train the model
svm_model.fit(X_train, y_train)

# ---------------------------------------------------
# MAKE PREDICTIONS
# ---------------------------------------------------

y_pred = svm_model.predict(X_test)



In [20]:

# Evaluation of SVM Classifier


# Evaluates the model using:  - Confusion matrix, Accuracy, Precision


from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)


cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

print("Confusion Matrix:")
print(cm)

print("\nAccuracy:", round(accuracy, 4))
print("Precision (Malignant):", round(precision, 4))
print("Recall / Sensitivity (Malignant):", round(recall, 4))
print("F1-score (Malignant):", round(f1, 4))

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

Confusion Matrix:
[[6 0]
 [0 7]]

Accuracy: 1.0
Precision (Malignant): 1.0
Recall / Sensitivity (Malignant): 1.0
F1-score (Malignant): 1.0

Detailed Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00         6
   Malignant       1.00      1.00      1.00         7

    accuracy                           1.00        13
   macro avg       1.00      1.00      1.00        13
weighted avg       1.00      1.00      1.00        13

